In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool


@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x**0.5


@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x**2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(model="claude-haiku-4-5", tools=[square_root])

subagent_2 = create_agent(model="claude-haiku-4-5", tools=[square])

## Calling subagents

In [4]:
from langchain.messages import HumanMessage


@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content


@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content


## Creating the main agent

main_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.",
)

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='9d093470-2489-44fa-a2a7-6d48efba03b9'),
              AIMessage(content=[{'id': 'toolu_014twUEZx1ixA5MP86HY5w9m', 'caller': {'type': 'direct'}, 'input': {'x': 456}, 'name': 'call_subagent_1', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01Q7btQEaxGLLSHtUAxiFEZW', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 672, 'output_tokens': 58, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019d876a-fd7c-7cc0-9863-542b85175356-0', tool_calls=[{'name': 'call_subagent_1', 'arg